# Bài 7: Chơi Tic-Tac-Toe Tương tác (Interactive Player)

Notebook này hiện thực hóa tác tử tương tác `interactive_player()`, cho phép người chơi trực tiếp nhập nước đi (từ 0 đến 8) từ bàn phím để đối đầu với các tác tử AI đã xây dựng (như `alpha_beta_player` hay `random_player`).

Đồng thời, notebook tiến hành **chạy và ghi lại chi tiết các ván đấu hoàn chỉnh ở cả hai vai X và O**, kèm nhận xét chiến lược chuyên sâu của tác tử AI.

In [1]:
import time
from unittest.mock import patch
from tictactoe import (
    empty_board, actions, result, terminal, utility, other,
    show_board_text, alpha_beta_player, play, random_player
)
print('Các module cốt lõi đã sẵn sàng.')

Các module cốt lõi đã sẵn sàng.


In [2]:
import numpy as np

def show_interactive_board(board):
    """Hiển thị bàn cờ 3x3: các ô đã đánh hiển thị 'x'/'o', các ô trống hiển thị chỉ số (0-8)"""
    display_cells = [board[i] if board[i] != ' ' else str(i) for i in range(9)]
    print(np.array(display_cells).reshape(3, 3))

def interactive_player(board, player=None):
    """
    Tác tử tương tác với người dùng:
    - Hiển thị trực quan trạng thái bàn cờ hiện tại.
    - Liệt kê danh sách các ô hợp lệ còn trống.
    - Yêu cầu người chơi nhập chỉ số ô (0-8) hoặc -1 để thoát.
    - Kiểm tra và bắt lỗi đầu vào nghiêm ngặt.
    """
    show_interactive_board(board)
    available = actions(board)
    print(f'Nước đi hợp lệ khả dĩ: {available}')
    
    while True:
        try:
            raw = input('Nhập nước đi của bạn (0-8 hoặc -1 để kết thúc):\n').strip()
            move = int(raw)
            if move == -1:
                print('Trò chơi kết thúc bởi người dùng.')
                raise Exception('Game ended by user.')
            if move in available:
                return move
            print(f'Nước đi {move} không hợp lệ! Vui lòng chọn một trong các ô: {available}')
        except ValueError:
            print('Giá trị không hợp lệ! Vui lòng nhập một số nguyên từ 0 đến 8.')

print('Hàm interactive_player() đã được cài đặt thành công.')

Hàm interactive_player() đã được cài đặt thành công.


## 1. Ván đấu 1: Người chơi cầm X (đi trước) đối đầu với AI cầm O (đi sau)

Ở ván đấu này, ta gọi hàm chuẩn `play(interactive_player, alpha_beta_player, N=1)`.
Người chơi cầm quân **X** đi trước, chủ động chọn các nước mở cuộc vào góc (ô 0, ô 2,...). Ta ghi lại toàn bộ tiến trình tương tác của `interactive_player()` và phân tích phản ứng phòng thủ của AI.

In [3]:
# Kịch bản nước đi của Người chơi (X): Đánh góc 0 -> đe dọa góc 2 -> đánh đáy 7 -> 5 -> 6
human_moves_x = iter(['0', '2', '7', '5', '6'])
def mock_input_x(prompt):
    val = next(human_moves_x)
    print(f'>>> [Người chơi X nhập]: {val}')
    return val

print('=== BẮT ĐẦU VÁN ĐẤU 1: Human (X) vs AI (O) ===\n')
with patch('builtins.input', side_effect=mock_input_x):
    res_match_1 = play(interactive_player, alpha_beta_player, N=1)

print('\n' + '='*40)
print(f'KẾT QUẢ VÁN ĐẤU 1: {res_match_1}')
if res_match_1['x'] == 1: print('Chiến thắng: Người chơi (X) thắng!')
elif res_match_1['o'] == 1: print('Chiến thắng: AI (O) thắng!')
else: print('Chiến thắng: HÒA (Draw) - Cả hai bên đều chơi tối ưu!')

=== BẮT ĐẦU VÁN ĐẤU 1: Human (X) vs AI (O) ===

[['0' '1' '2']
 ['3' '4' '5']
 ['6' '7' '8']]
Nước đi hợp lệ khả dĩ: [0, 1, 2, 3, 4, 5, 6, 7, 8]
>>> [Người chơi X nhập]: 0
[['x' '1' '2']
 ['3' 'o' '5']
 ['6' '7' '8']]
Nước đi hợp lệ khả dĩ: [1, 2, 3, 5, 6, 7, 8]
>>> [Người chơi X nhập]: 2
[['x' 'o' 'x']
 ['3' 'o' '5']
 ['6' '7' '8']]
Nước đi hợp lệ khả dĩ: [3, 5, 6, 7, 8]
>>> [Người chơi X nhập]: 7
[['x' 'o' 'x']
 ['o' 'o' '5']
 ['6' 'x' '8']]
Nước đi hợp lệ khả dĩ: [5, 6, 8]
>>> [Người chơi X nhập]: 5
[['x' 'o' 'x']
 ['o' 'o' 'x']
 ['6' 'x' 'o']]
Nước đi hợp lệ khả dĩ: [6]
>>> [Người chơi X nhập]: 6

KẾT QUẢ VÁN ĐẤU 1: {'x': 0, 'o': 0, 'd': 1}
Chiến thắng: HÒA (Draw) - Cả hai bên đều chơi tối ưu!


## 2. Ván đấu 2: AI cầm X (đi trước) đối đầu với Người chơi cầm O (đi sau)

Ở ván đấu này, ta gọi hàm chuẩn `play(alpha_beta_player, interactive_player, N=1)`.
Tác tử AI cầm quân **X** đi trước, chủ động chiếm vị trí chiến lược. Người chơi cầm quân **O** đi sau và thử nghiệm một nước đi thiếu chính xác (không chiếm trung tâm mà đánh vào cạnh ô 1) để kiểm tra xem AI có nhận biết và trừng phạt sai lầm để giành chiến thắng hay không.

In [4]:
# Kịch bản nước đi của Người chơi (O): Lượt đầu đi ô 1 (sai sót), sau đó đi ô 8, ô 6
human_moves_o = iter(['1', '8', '6'])
def mock_input_o(prompt):
    val = next(human_moves_o)
    print(f'>>> [Người chơi O nhập]: {val}')
    return val

print('=== BẮT ĐẦU VÁN ĐẤU 2: AI (X) vs Human (O) ===\n')
with patch('builtins.input', side_effect=mock_input_o):
    res_match_2 = play(alpha_beta_player, interactive_player, N=1)

print('\n' + '='*40)
print(f'KẾT QUẢ VÁN ĐẤU 2: {res_match_2}')
if res_match_2['x'] == 1: print('Chiến thắng: AI (X) THẮNG THUYẾT PHỤC!')
elif res_match_2['o'] == 1: print('Chiến thắng: Người chơi (O) thắng!')
else: print('Chiến thắng: HÒA (Draw)!')

=== BẮT ĐẦU VÁN ĐẤU 2: AI (X) vs Human (O) ===

[['x' '1' '2']
 ['3' '4' '5']
 ['6' '7' '8']]
Nước đi hợp lệ khả dĩ: [1, 2, 3, 4, 5, 6, 7, 8]
>>> [Người chơi O nhập]: 1
[['x' 'o' '2']
 ['x' '4' '5']
 ['6' '7' '8']]
Nước đi hợp lệ khả dĩ: [2, 4, 5, 6, 7, 8]
>>> [Người chơi O nhập]: 8
[['x' 'o' '2']
 ['x' 'x' '5']
 ['6' '7' 'o']]
Nước đi hợp lệ khả dĩ: [2, 5, 6, 7]
>>> [Người chơi O nhập]: 6

KẾT QUẢ VÁN ĐẤU 2: {'x': 1, 'o': 0, 'd': 0}
Chiến thắng: AI (X) THẮNG THUYẾT PHỤC!


## 3. Nhận xét & Đánh giá Chiến lược của Tác tử AI ở cả hai vai X và O

Dựa trên các ván đấu đã ghi lại trực tiếp ở trên:

### 3.1. Chiến lược khi AI cầm quân O (Đi sau - Phòng thủ):
1. **Chiếm cứ điểm trung tâm ngay lập tức:** Khi Người chơi (X) mở màn đánh vào góc (ô 0), AI không ngần ngại chiếm ngay ô trung tâm (ô 4). Ô trung tâm là vị trí then chốt có giá trị phòng thủ cao nhất vì nó kiểm soát 4 đường thẳng (2 đường chéo, 1 hàng ngang, 1 cột dọc).
2. **Chặn đứng mọi nguy cơ tạo 3 quân liên tiếp:** Khi Người chơi đánh tiếp ô 2 (đe dọa hàng trên `0 - 1 - 2`), AI lập tức phản ứng bằng nước chặn ô 1, bẻ gãy hoàn toàn mũi tấn công của đối thủ.
3. **Triệt tiêu thế cờ đôi (Fork Prevention):** AI liên tục chọn các nước đi tối ưu ở các ô 8 và 3, ép Người chơi vào thế phải đỡ và đưa ván cờ về kết quả **Hòa (Draw)** tuyệt đối.

### 3.2. Chiến lược khi AI cầm quân X (Đi trước - Tấn công):
1. **Mở cuộc chủ động:** AI mở màn bằng ô góc (ô 0), đây là nước đi mở tạo ra nhiều biến thể tấn công nhất trong lý thuyết Tic-Tac-Toe.
2. **Trừng phạt sai lầm chiến thuật của đối thủ:** Khi Người chơi (O) mắc sai lầm chọn nước đi yếu (ô 1 thay vì chiếm trung tâm ô 4), AI lập tức chớp thời cơ mở rộng tầm ảnh hưởng sang ô 3 và ô 4.
3. **Tạo thế bẫy hai mang (Fork):** Ở nước thứ 3, AI tạo ra cùng lúc hai mối đe dọa chiến thắng (hàng dọc `0 - 3 - 6` và hàng ngang `3 - 4 - 5`). Khi Người chơi chỉ có thể chặn được một hướng (ô 6), AI lập tức đánh ô 5 ở nước kế tiếp để **kết liễu ván đấu với chiến thắng tuyệt đối**.

In [5]:
# ==========================================================================
# KHU VỰC CHƠI TRỰC TIẾP (LIVE INTERACTIVE PLAY)
# ==========================================================================
# Nếu bạn đang mở Notebook này trong giao diện JupyterLab / VS Code và muốn
# tự tay gõ phím trực tiếp đối đầu với AI, hãy bỏ dấu comment (#) ở 1 trong 2 dòng dưới đây:

# Bạn cầm X (đi trước) đấu với Alpha-Beta AI:
# play(interactive_player, alpha_beta_player, N=1)

# AI cầm X (đi trước) đấu với Bạn cầm O (đi sau):
# play(alpha_beta_player, interactive_player, N=1)